# What is function calling (in the context of language models)? 

"Function calling" in language models  (LLMs or SLMs) refers to the ability for the model to interpret user input and choose between predefined functions to perform actions, retrieve data, or interact with external systems.

IMPORTANT: the language model does not actually execute the function. Rather, it determines which function should be used (if any) as well as the values of the parameters for the function.  ie. it only generates a structured function call (often in JSON format) with the function name and parameters.

### How function-calling works - example: 

1. User inputs a request such as: “What’s the weather in Berlin?”

2. Instead of generating a text response, the language model (LM) determines that it needs to use the 'get_weather' function to retrieve weather data and returns a structured function call (JSON format)

    ```json 
    {
        "function": "get_weather",
        "parameters": {
            "location": "New York",
            "unit": "Celsius"
        }
    }
    ```

3. Your application then executes the function.  It can then either: 
    - Format the output and return it to the user, OR  
    - Send the output of the function, together with the original user request back to the LM to format a response.  



🤔 "But how does the LM even know that the function 'get_weather()' exists or what the valid arguments are?", you ask. 

Great question! We're coming to that next... 

### How does the language model know which functions are available? 

You must explicitly provide a list of functions and their parameters when making a request. The model only chooses from the functions you define in the input. These functions are often (but not always) described in a structured format, like JSON schemas.

You may be familiar with (Azure) OpenAI APIs where you can extend the capabilities of OpenAI models by giving them access to tools, which can have one of two forms:

https://platform.openai.com/docs/guides/function-calling

![openai-function-calling](images/openai-function-calling.png)

Btw, function calling & tools are the backbone of Agents. 💡

### 🤔 QUESTION: How can we provide JSON schema describing our functions to the SLM ?

Our SLM doesn't have an API which allows the function JSON schema to be defined separately.  So how do we instruct the model? 

<details>
    <summary>Click to answer</summary>
    
By describing the available functions in the system prompt. 

</details>


## Exercise: instruct the SLM in function-calling

Imagine you have the following functions available

> 1. Climate control
>     - `set_temperature(degrees: int, unit: Optional[str])` – Adjusts the temperature in the car (in C/F).
>     - `toggle_ac(state: bool)` – Turns the air conditioning on or off.
>    - `set_fan_speed(level: int)` – Adjusts the fan speed.
> 
> 2. Music & Media
>     - `play_music(artist: Optional[str], song: Optional[str], genre: Optional[str])` – Plays music based on request.
> 	  - `adjust_volume(level: int)` – Sets the volume to a specific level.
> 	  - `skip_track()` – Skips to the next song.
> 	  - `pause_music()` – Pauses the currently playing music.
> 
> 3. Navigation & Maps
> 	  - `set_destination(location: str)` – Sets a GPS destination.
> 	  - `get_estimated_arrival_time()` – Returns estimated time of arrival.
> 	  - `reroute()` – Finds an alternative route.
> 	
> 4. Phone & Communication
>     - `call_contact(name: str)` – Calls a saved contact.
>     - `send_text(contact: str, message: str)` – Sends a text message.
>     - `read_last_text()` – Reads the most recent text message.

If you had to write a system prompt for your SLM informing it of the available functions, what should you mention or include in the system prompt? 

For example: 
1. context (overview of the use case) 
2. describe the functions 
2. ... 

What else?  

<details>
    <summary>Click for one possible answer</summary>

Some things to consider including in your system prompt: 

1. Context describing the use-case
2. Describe the response format 
3. Describe the available functions
4. Response guidelines 
5. Example inputs/outputs
6. Error handling 
7. How to provide a non-function response 

Here is an [hypothetical prompt](assets/03-function-calling-prompt.md)

Notice that that prompt template does not use JSON schema in the interest of size (JSON requires more tokens but can be more explicit). 

</details>

## Exercise: use a Hugging Face pipeline to test your instructions to the SLM 

Using the transformers library to load a model, tokenizer & pipeline - as you learned in the last chapter - try testing the hypothetical system prompt (answer above👆) or your own prompt with some prompts, to see if the model can do function calling based on prompt instructions alone. 

You've seen the following code before: 

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline 

model_name  = "microsoft/Phi-3.5-mini-instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # device_map="auto",
    # torch_dtype="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

Now load our prompt and create an inference function: 

In [ ]:
from functools import partial 

system_prompt = """ """   # <-- Try adding your own prompt here.   (Or uncomment the lines below to use the one prepared)

# with open("assets/03-function-calling-prompt.md", "r") as file:
#     system_prompt = file.read()


def run_pipeline(pipe, user_query):
    """Function to run inference"""

    # torch.random.manual_seed(0)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query},
    ]

    output = pipe(
        messages,
        max_new_tokens=100,
        return_full_text=False,
        do_sample=False,
    )

    print("\n".join([
        "----- USER INPUT:",
        user_query,
        "",
        "----- RESPONSE:",
        output[0]['generated_text'].strip(),
        ])
    )

# Create a new function called 'run_pipe' which is equivalent to the 'run_inference" function 
# but with the pipeline prefilled. 
infer = partial(run_pipeline, pipe)

Test your inference function - try changing the text and run the cell: 

In [ ]:
infer("make it louder please")

If you are finding that it takes an inordinately long time and you have an Nvidia GPU, you may wish to take a look at the [optimization options in notebook 7](./07-model-and-tokenizer-config.ipynb). 